# Chapter 22: Advanced Optimization Methods

This notebook explores **multi-objective optimization** and **scenario comparison**
for production optimization. We demonstrate Pareto front visualization for
conflicting objectives (maximize production vs minimize power) and compare
summer vs winter operating conditions.

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import importlib, subprocess, sys

try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False)
    ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
    print("NeqSim loaded via devtools (local dev mode)")
except Exception:
    NEQSIM_MODE = "pip"

# Always ensure jneqsim is available (works in both modes)
try:
    import neqsim
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])

from neqsim import jneqsim
print(f"NeqSim ready (mode: {NEQSIM_MODE})")

# Common class shortcuts for convenience
SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
SystemPrEos = jneqsim.thermo.system.SystemPrEos
SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

# Process equipment
Stream = jneqsim.process.equipment.stream.Stream
Separator = jneqsim.process.equipment.separator.Separator
ThreePhaseSeparator = jneqsim.process.equipment.separator.ThreePhaseSeparator
Compressor = jneqsim.process.equipment.compressor.Compressor
Cooler = jneqsim.process.equipment.heatexchanger.Cooler
Heater = jneqsim.process.equipment.heatexchanger.Heater
HeatExchanger = jneqsim.process.equipment.heatexchanger.HeatExchanger
Mixer = jneqsim.process.equipment.mixer.Mixer
Splitter = jneqsim.process.equipment.splitter.Splitter
ThrottlingValve = jneqsim.process.equipment.valve.ThrottlingValve
Pump = jneqsim.process.equipment.pump.Pump
Expander = jneqsim.process.equipment.expander.Expander
Recycle = jneqsim.process.equipment.util.Recycle
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim2
Classpath:
  1. C:\Users\ESOL\Documents\GitHub\neqsim2\target\classes
  2. C:\Users\ESOL\Documents\GitHub\neqsim2\src\main\resources
  3. C:\Users\ESOL\Documents\GitHub\neqsim2\target\neqsim-3.7.0.jar



JVM started: C:\Users\ESOL\graalvm\graalvm-jdk-25.0.1+8.1\bin\server\jvm.dll
Ready — call neqsim_classes(ns) to import classes


All NeqSim classes imported OK
NeqSim loaded via devtools (local dev mode)
NeqSim ready (mode: devtools)


## 22.1 Multi-Objective Optimization: Concept

In production optimization, we often face **conflicting objectives**:

- **Maximize production rate** (revenue)
- **Minimize compressor power** (operating cost)
- **Minimize flaring/emissions**

No single operating point can optimize all objectives simultaneously. Instead,
we seek the **Pareto front** — the set of solutions where improving one objective
necessarily worsens another.

In [2]:
# Build a simple compression process at different throughputs
fluid = jneqsim.thermo.system.SystemSrkEos(273.15 + 30.0, 50.0)
fluid.addComponent("methane", 0.85)
fluid.addComponent("ethane", 0.08)
fluid.addComponent("propane", 0.04)
fluid.addComponent("CO2", 0.02)
fluid.addComponent("nitrogen", 0.01)
fluid.setMixingRule("classic")

flow_rates = np.linspace(30000, 120000, 10)  # kg/hr
comp_powers = []

for flow in flow_rates:
    feed = jneqsim.process.equipment.stream.Stream("Feed", fluid.clone())
    feed.setFlowRate(float(flow), "kg/hr")
    feed.setTemperature(30.0, "C")
    feed.setPressure(50.0, "bara")

    comp = jneqsim.process.equipment.compressor.Compressor("Compressor", feed)
    comp.setOutletPressure(120.0)

    proc = jneqsim.process.processmodel.ProcessSystem()
    proc.add(feed)
    proc.add(comp)
    proc.run()

    comp_powers.append(comp.getPower('kW'))

comp_powers = np.array(comp_powers)
print(f"Flow range: {flow_rates[0]/1000:.0f} - {flow_rates[-1]/1000:.0f} t/hr")
print(f"Power range: {comp_powers[0]/1000:.1f} - {comp_powers[-1]/1000:.1f} MW")

Flow range: 30 - 120 t/hr
Power range: 1.0 - 3.9 MW


## 22.2 Generate Pareto Front

We generate a Pareto front combining NeqSim results with synthetic variation
to illustrate the trade-off between production rate and specific power consumption.

In [3]:
np.random.seed(42)

# Production rate in MSm3/d (proportional to mass flow)
production = flow_rates / 1000.0 * 0.035  # approximate conversion

# Specific power (kW per unit production)
specific_power = comp_powers / production

# Generate dominated (non-optimal) solutions
n_dominated = 40
dom_prod = np.random.uniform(production.min(), production.max(), n_dominated)
dom_power = np.interp(dom_prod, production, specific_power) * np.random.uniform(1.05, 1.4, n_dominated)

# Pareto front points (from NeqSim results)
pareto_prod = production
pareto_power = specific_power

print(f"Pareto front: {len(pareto_prod)} points")
print(f"Dominated solutions: {n_dominated} points")

Pareto front: 10 points
Dominated solutions: 40 points


In [4]:
fig, ax = plt.subplots(figsize=(10, 7))

# Dominated solutions
ax.scatter(dom_prod, dom_power, c='lightgray', s=40, alpha=0.6,
           edgecolors='gray', label='Dominated solutions')

# Pareto front
sorted_idx = np.argsort(pareto_prod)
ax.plot(pareto_prod[sorted_idx], pareto_power[sorted_idx], 'r-o',
        linewidth=2.5, markersize=8, label='Pareto front', zorder=5)

# Highlight trade-off region
ax.annotate('High production,\nhigh specific power',
            xy=(pareto_prod[-1], pareto_power[-1]),
            xytext=(pareto_prod[-1]-0.3, pareto_power[-1]+50),
            fontsize=9, ha='right',
            arrowprops=dict(arrowstyle='->', color='black'))

ax.annotate('Low production,\nlow specific power',
            xy=(pareto_prod[0], pareto_power[0]),
            xytext=(pareto_prod[0]+0.5, pareto_power[0]-40),
            fontsize=9,
            arrowprops=dict(arrowstyle='->', color='black'))

ax.set_xlabel('Production Rate [MSm³/d]', fontsize=12)
ax.set_ylabel('Specific Compressor Power [kW/(MSm³/d)]', fontsize=12)
ax.set_title('Multi-Objective Optimization: Pareto Front\n(Production vs Compression Efficiency)', fontsize=13)
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../figures/ch22_pareto_front.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to ../figures/ch22_pareto_front.png")

Figure saved to ../figures/ch22_pareto_front.png


C:\Users\ESOL\AppData\Local\Temp\ipykernel_39328\2370357463.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 22.3 Scenario Comparison: Summer vs Winter

Ambient temperature affects cooling capacity and compressor power. We compare
key performance indicators for summer (30 °C) and winter (5 °C) conditions.

In [5]:
scenarios = {
    "Summer (30 C)": {"ambient_T": 30.0, "cooling_T": 40.0},
    "Winter (5 C)": {"ambient_T": 5.0, "cooling_T": 15.0},
}

results = {}

for name, params in scenarios.items():
    s_fluid = jneqsim.thermo.system.SystemSrkEos(273.15 + params["ambient_T"], 50.0)
    s_fluid.addComponent("methane", 0.85)
    s_fluid.addComponent("ethane", 0.08)
    s_fluid.addComponent("propane", 0.04)
    s_fluid.addComponent("CO2", 0.02)
    s_fluid.addComponent("nitrogen", 0.01)
    s_fluid.setMixingRule("classic")

    s_feed = jneqsim.process.equipment.stream.Stream("Feed", s_fluid)
    s_feed.setFlowRate(80000.0, "kg/hr")
    s_feed.setTemperature(float(params["ambient_T"]), "C")
    s_feed.setPressure(50.0, "bara")

    s_comp = jneqsim.process.equipment.compressor.Compressor("Compressor", s_feed)
    s_comp.setOutletPressure(120.0)

    s_cooler = jneqsim.process.equipment.heatexchanger.Cooler("Cooler", s_comp.getOutletStream())
    s_cooler.setOutTemperature(273.15 + params["cooling_T"])

    s_proc = jneqsim.process.processmodel.ProcessSystem()
    s_proc.add(s_feed)
    s_proc.add(s_comp)
    s_proc.add(s_cooler)
    s_proc.run()

    results[name] = {
        "Compressor Power [MW]": s_comp.getPower('kW') / 1000.0,
        "Discharge T [C]": s_comp.getOutletStream().getTemperature('C'),
        "Cooler Duty [MW]": abs(s_cooler.getDuty() / 1e6),
        "Export T [C]": s_cooler.getOutletStream().getTemperature('C'),
    }

    print(f"\n{name}:")
    for k, v in results[name].items():
        print(f"  {k}: {v:.2f}")


Summer (30 C):
  Compressor Power [MW]: 2.60
  Discharge T [C]: 97.34
  Cooler Duty [MW]: 3.67
  Export T [C]: 40.00

Winter (5 C):
  Compressor Power [MW]: 2.28
  Discharge T [C]: 70.13
  Cooler Duty [MW]: 3.78
  Export T [C]: 15.00


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Bar chart: scenario comparison
metrics = list(list(results.values())[0].keys())
scenario_names = list(results.keys())
x = np.arange(len(metrics))
width = 0.35

summer_vals = [results[scenario_names[0]][m] for m in metrics]
winter_vals = [results[scenario_names[1]][m] for m in metrics]

axes[0].bar(x - width/2, summer_vals, width, label=scenario_names[0], color='#e15759', alpha=0.8)
axes[0].bar(x + width/2, winter_vals, width, label=scenario_names[1], color='#4e79a7', alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels([m.replace(' [', '\n[') for m in metrics], fontsize=9)
axes[0].set_ylabel('Value')
axes[0].set_title('Summer vs Winter: Process KPIs', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Percentage difference
pct_diff = [(w - s) / s * 100 for s, w in zip(summer_vals, winter_vals)]
colors = ['green' if p < 0 else 'red' for p in pct_diff]
axes[1].barh(metrics, pct_diff, color=colors, alpha=0.7, edgecolor='black', linewidth=0.5)
axes[1].axvline(x=0, color='black', linewidth=1)
axes[1].set_xlabel('Change from Summer to Winter [%]')
axes[1].set_title('Winter vs Summer: Relative Change', fontsize=12)
axes[1].grid(True, alpha=0.3, axis='x')

plt.suptitle('Seasonal Operating Scenario Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("../figures/ch22_scenario_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to ../figures/ch22_scenario_comparison.png")

Figure saved to ../figures/ch22_scenario_comparison.png


C:\Users\ESOL\AppData\Local\Temp\ipykernel_39328\3897581305.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 22.4 Summary

**Key takeaways:**

1. **Multi-objective optimization** reveals trade-offs that single-objective methods miss.
2. The **Pareto front** shows the set of non-dominated solutions — no improvement in one
   objective is possible without worsening another.
3. **Scenario comparison** (summer vs winter) quantifies the impact of ambient conditions
   on plant performance — winter typically reduces compressor power due to colder inlet gas.
4. Decision-makers select operating points on the Pareto front based on economic and
   operational priorities.
5. NeqSim enables rapid evaluation of many operating scenarios for optimization studies.